In [1]:
import os
os.chdir('../../../../..')

In [2]:
from src.datasets import MaterialsProject

In [3]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.helper_functions import create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [4]:
mp = MaterialsProject(limit=5000, sampling_strategy="stratified", stratify_on=["band_gap", "energy_above_hull"], descriptors=["acsf"])
df = mp.load()

2026-04-28 14:52:35.460 | INFO     | src.datasets:load:1979 - Loading full cached Parquet data from data/Materials Project/materials.parquet...
2026-04-28 14:52:36.436 | INFO     | src.datasets:_add_descriptors:2352 - Ignoring output_tag=sample_n6000_seed40_stratified since descriptors are attached directly to dataframe.
2026-04-28 14:52:36.437 | INFO     | src.datasets:_add_descriptors:2356 - Extracting unique elements from formulas...
2026-04-28 14:52:36.463 | INFO     | src.datasets:_add_descriptors:2364 - Found 86 unique elements.
2026-04-28 14:52:36.464 | INFO     | src.datasets:_add_descriptors:2425 - Computing ACSF chunk 0 (0 to 1000)...
2026-04-28 14:52:47.548 | INFO     | src.datasets:_add_descriptors:2425 - Computing ACSF chunk 1 (1000 to 2000)...
2026-04-28 14:53:05.175 | INFO     | src.datasets:_add_descriptors:2425 - Computing ACSF chunk 2 (2000 to 3000)...
2026-04-28 14:53:17.141 | INFO     | src.datasets:_add_descriptors:2425 - Computing ACSF chunk 3 (3000 to 4000)...
20

: 

In [ ]:
dist_matrix = mp.get_distance_matrix(
    descriptor="acsf",
    dist_type="euclidean",
    force_calculate=True
)

# Hierarchical Clustering on Distance Matrix

In [ ]:
hier = AgglomerativeClustering(n_clusters=3, metric='precomputed', linkage='complete')
labels_hier = hier.fit_predict(dist_matrix)
df = df.with_columns(labels_hier=labels_hier)

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_hier,
    clustering_method="hierarchical"
)

In [ ]:
average_numeric_by_cluster(df, "labels_hier")

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

# KMedoids

In [ ]:
model_km = KMedoids(n_clusters=4, metric="precomputed", linkage='complete')
labels_km = model_km.fit_predict(dist_matrix)
print(np.unique(labels_km, return_counts=True))
df = df.with_columns(labels_km=labels_km)

In [ ]:
average_numeric_by_cluster(df, "labels_kmedoids")

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_km,
    clustering_method="kmedoids"
)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

# Spectral

In [ ]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_spectral,
    clustering_method="spectral"
)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

In [ ]:
average_numeric_by_cluster(df, "labels_spectral")

# DBSCAN

In [ ]:
model_db = DBSCAN(
    eps=5,
    min_samples=3,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db,return_counts=True))

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_db,
    clustering_method="dbscan"
)

In [ ]:
average_numeric_by_cluster(df, "labels_db")

# KMeans on Raw Embeddings

In [ ]:
X_raw = np.array(df["acsf_embedding"].to_list(), dtype=np.float32)
kmeans_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans_raw = kmeans_raw.fit_predict(X_raw)
df = df.with_columns(labels_kmeans_raw=labels_kmeans_raw)

In [ ]:
create_chemiscope_viewer(df, X_raw, labels_kmeans_raw, 'PCA')

In [ ]:
average_numeric_by_cluster(df, "labels_kmeans_raw")